# TreeCache — Aggregation Tree Cache

Complete binary tree, flat array representation.  Manages the φ
bijection (token ↔ node index).  Pure local data structure — no
network operations.

**States** per node:
- ``None`` — ⊥ (dirty, needs recomputation)
- ``NOT_LOADED`` — clean, on disk
- ``ShareVec`` — clean, in memory


In [ ]:
import mpmt
from mpmt.tree_cache import TreeCache, MergeStep, NOT_LOADED

tc = TreeCache(max_holders=4)
print(f"height={tc.height}, max_nodes={tc.max_nodes}")


## insert(token, share)

First holder → placed at root (index 0).

Subsequent holders → *top-down splitting*: the node at index
``c = count - 1`` is split — its old content moves to the left child
(2c+1), the new share goes to the right child (2c+2), and the parent
*c* is marked ⊥.


In [ ]:
tc.insert(token=b"holder_0", share=None)
tc.insert(token=b"holder_1", share=None)
tc.insert(token=b"holder_2", share=None)
assert tc.count == 3

# After 3 inserts, leaves are at indices 3, 4, 5
print(f"leaves: {tc.leaf_indices()}")


## update(token, new_share)

Replace the share at the leaf identified by *token*, then mark all
ancestors ⊥ (path to root).


In [ ]:
tc.update(token=b"holder_1", new_share=None)
print(f"dirty: {tc.dirty_nodes()}")


## remove(token)

*Heap-style deletion*: swap with the last active leaf, remove the
last position, update φ, mark both affected paths ⊥.
Preserves tree balance under repeated removals.


In [ ]:
tc.remove(token=b"holder_0")
assert tc.count == 2
assert tc.index_of(token=b"holder_0") is None
print(f"remaining: {tc.leaf_indices()}")


## get_merge_schedule() → list[MergeStep]

Bottom-up scan of all internal nodes.  For each ⊥ node whose children
are available (clean or already scheduled), a ``MergeStep(parent=...)``
is emitted.  The caller executes each step using its Rep3 instance.

**Pass-through**: if only one child is available (after a quit),
the merge executor copies the surviving child's value into the parent.


In [ ]:
sched = tc.get_merge_schedule()
for s in sched:
    print(f"parent={s.parent}, children=({2*s.parent+1},{2*s.parent+2})")


## Properties and helpers

- ``tc.count`` — active holders
- ``tc.height``, ``tc.max_nodes``
- ``tc.root_share`` — B(∪X_i), or None if not yet computed
- ``tc.index_of(token)`` → int | None
- ``tc.token_of(idx)`` → bytes | None
- ``tc.share_at(idx)`` → None | NOT_LOADED | ShareVec
- ``tc.leaf_indices()`` — sorted active leaf indices
- ``tc.dirty_nodes()`` — ⊥ nodes (debug aid)
- ``NOT_LOADED is not None`` — sentinel for disk-resident shares


## MergeStep

``MergeStep(parent: int)`` — a frozen dataclass.  Left child =
2*parent+1, right child = 2*parent+2.  The caller doesn't need to
store child indices — they're derived from the parent.
